### **STEP 1.3 - Group Regulatory Families & Split Dataset**

**Input:** `outputs/logs/corpus_registry.json`, cleaned `.txt` files and the relationship CSV.

**Output:** development/eval/test folders, `split_manifest.json` and a split report.


In [18]:
import json
import random
import re
import shutil

import pandas as pd

from collections import Counter, defaultdict
from pathlib import Path


In [20]:
# Configure folders used by the split notebook
PROJECT_ROOT = Path("/Users/tanggiee/Desktop/RAG_AI/esg_rag_project")
LOG_FOLDER = PROJECT_ROOT / "outputs" / "logs"
MANIFEST_FOLDER = PROJECT_ROOT / "outputs" / "manifests"
REPORT_FOLDER = PROJECT_ROOT / "outputs" / "reports"

SPLIT_FOLDER = PROJECT_ROOT / "data" / "splits"
SPLIT_DEV = SPLIT_FOLDER / "development"
SPLIT_EVAL = SPLIT_FOLDER / "eval"
SPLIT_TEST = SPLIT_FOLDER / "test"

for folder in [MANIFEST_FOLDER, REPORT_FOLDER, SPLIT_DEV, SPLIT_EVAL, SPLIT_TEST]:
    folder.mkdir(parents=True, exist_ok=True)


In [21]:
# Load the completed registry created in Step 1.2
REGISTRY_PATH = LOG_FOLDER / "corpus_registry.json"
registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
registry_df = pd.DataFrame(registry)

# Keep successful records whose cleaned text still exists
docs = [doc for doc in registry if doc.get("status") == "ok"
        and doc.get("cleaned_path") and Path(doc["cleaned_path"]).exists()]

print("Registry records:", len(registry))
print("Documents available:", len(docs))
print("Missing cleaned files:", len(registry) - len(docs))


Registry records: 403
Documents available: 403
Missing cleaned files: 0


In [22]:
# Split ratios and reproducibility
DEV_RATIO, EVAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
RANDOM_SEED = 42
SPLIT_RATIOS = {"development": DEV_RATIO, "eval": EVAL_RATIO, "test": TEST_RATIO}


#### **1. Detect direct regulatory relationships**

Only amendment, replacement, consolidation and similar legal-lineage relations form families. Ordinary citations such as `refer_to`, `based_on` and `guided_by` do not.


In [23]:
# Normalize legal numbers so different separators still match
def normalize_number(value: str) -> str:
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


# Detect referenced documents and direct relationship words in titles
REFERENCE_PATTERN = re.compile(
    r"\b(?:DECREE|CIRCULAR|DECISION|LAW|RESOLUTION|ORDINANCE)"
    r"\s+(?:NO[.\s]*)?([0-9][A-Z0-9/_.-]*)", re.IGNORECASE)

RELATION_PATTERN = re.compile(
    r"\b(?:amend|amending|amendments?|supplement|supplementing|replace|"
    r"replacing|supersede|superseding|consolidate|consolidating|integrate|"
    r"integrated|rectify|rectifying)\b", re.IGNORECASE)


In [24]:
# Join documents into non-overlapping regulatory families
class UnionFind:
    def __init__(self, items):
        self.parent = {item: item for item in items}

    def find(self, item):
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])
        return self.parent[item]

    def union(self, first, second):
        first_root, second_root = self.find(first), self.find(second)
        if first_root != second_root:
            self.parent[second_root] = first_root


In [26]:
# Start with one family per document and map legal numbers to doc_id
union_find = UnionFind([doc["doc_id"] for doc in docs])
number_lookup = {normalize_number(doc["official_number"]): doc["doc_id"] for doc in docs}

# Add direct relationships detected from titles
for doc in docs:
    title = doc.get("title", "")

    for reference in REFERENCE_PATTERN.finditer(title):
        preceding_text = title[max(0, reference.start() - 120):reference.start()]

        if not RELATION_PATTERN.search(preceding_text):
            continue

        related_doc_id = number_lookup.get(normalize_number(reference.group(1)))
        if related_doc_id:
            union_find.union(doc["doc_id"], related_doc_id)


#### **2. Add verified relationships from the CSV**

In [27]:
# Load and clean the manually collected relationship table
RELATIONSHIP_PATH = Path("/Users/tanggiee/Desktop/RAG_AI/ESG law dataset_relations.csv")
relationships_df = pd.read_csv(RELATIONSHIP_PATH).dropna(how="all")
relationships_df = relationships_df.rename(columns={"source_document_id (S)": "source_document_id"})
relationships_df["source_document_id"] = relationships_df["source_document_id"].ffill()
relationships_df = (relationships_df[["source_document_id", "relation_type", "target_official_number"]]
                    .dropna().reset_index(drop=True))
relationships_df["relation_type"] = relationships_df["relation_type"].str.strip().str.lower()

print("Valid relationship records:", len(relationships_df))
display(relationships_df.head())


Valid relationship records: 3130


,source_document_id,relation_type,target_official_number
0,119/2025/ND-CP,guide,72/2020/QH14
1,119/2025/ND-CP,based_on,63/2025/QH15
2,119/2025/ND-CP,amend,06/2022/ND-CP
3,119/2025/ND-CP,guided_by,11/2026/TT-BNNMT
4,119/2025/ND-CP,consolidated_in,10/VBHN-BNNMT


In [28]:
# Use only direct legal-lineage relationships for family grouping
FAMILY_RELATION_TYPES = {"amend", "amended_by", "replace", "replaced_by",
                         "consolidated_in", "supersede", "superseded_by",
                         "rectify", "rectified_by"}

family_relationships = relationships_df[
    relationships_df["relation_type"].isin(FAMILY_RELATION_TYPES)]

matched_relationships = 0

for _, relation in family_relationships.iterrows():
    source_id = number_lookup.get(normalize_number(relation["source_document_id"]))
    target_id = number_lookup.get(normalize_number(relation["target_official_number"]))

    if source_id and target_id:
        union_find.union(source_id, target_id)
        matched_relationships += 1

print("Legal-lineage relationships:", len(family_relationships))
print("Matched within corpus:", matched_relationships)


Legal-lineage relationships: 1092
Matched within corpus: 228


In [29]:
# Optional manually confirmed families not found from titles or the CSV
MANUAL_REGULATORY_FAMILIES = [
    # ["119/2025/ND-CP", "06/2022/ND-CP", "10/VBHN-BNNMT"]
]

for legal_family in MANUAL_REGULATORY_FAMILIES:
    family_doc_ids = [number_lookup.get(normalize_number(number)) for number in legal_family]
    family_doc_ids = [doc_id for doc_id in family_doc_ids if doc_id]

    for doc_id in family_doc_ids[1:]:
        union_find.union(family_doc_ids[0], doc_id)


#### **3. Build and review the final families**

In [30]:
# Collect documents sharing the same Union-Find root
root_families = defaultdict(list)

for doc in docs:
    root_families[union_find.find(doc["doc_id"])].append(doc)

# Assign stable family IDs
families = {}

for number, family_docs in enumerate(
        sorted(root_families.values(), key=lambda items: min(doc["doc_id"] for doc in items)), start=1):
    family_id = f"family_{number:04}"

    for doc in family_docs:
        doc["regulatory_family_id"] = family_id

    families[family_id] = family_docs

singletons = sum(len(family) == 1 for family in families.values())
multi = sum(len(family) > 1 for family in families.values())

print("Documents:", len(docs))
print("Regulatory families:", len(families))
print("Singleton families:", singletons)
print("Multi-document families:", multi)


Documents: 403
Regulatory families: 273
Singleton families: 244
Multi-document families: 29


In [31]:
# Show relationship-like titles still left as singleton families
unresolved_family_candidates = [{"official_number": doc["official_number"],
                                 "document_type": doc["document_type"],
                                 "title": doc["title"]}
                                for family in families.values() for doc in family
                                if len(family) == 1 and RELATION_PATTERN.search(doc.get("title", ""))]

display(pd.DataFrame(unresolved_family_candidates))
print("Unresolved family candidates:", len(unresolved_family_candidates))


,official_number,document_type,title
0,01/2022/TT-BNNPTNT,Circular,AMENDMENTS TO CIRCULARS IN AQUACULTURE
1,07/2022/ND-CP,Decree,PROVIDING AMENDMENTS TO DECREES ON PENALTIES F...
2,100/2016/ND-CP,Decree,ELABORATION AND GUIDELINES FOR SOME ARTICLES O...
3,106/2016/QH13,Law,AMENDMENTS TO SOME ARTICLES OF THE LAW ON VALU...
4,116/2018/ND-CP,Decree,AMENDING SEVERAL ARTICLES OF DECREE NO.55/2015...
5,123/2018/ND-CP,Decree,AMENDING AND SUPPLEMENTING CERTAIN DECREES ON ...
6,156/2025/ND-CP,Decree,ON AMENDMENTS TO DECREE NO. 55/2015/ND-CP DATE...
7,17/2022/ND-CP,Decree,AMENDMENT TO DECREES IMPOSING PENALTIES FOR AD...
8,17/2022/TT-BNNPTNT,Circular,AMENDMENT TO CIRCULAR NO. 29/2018/TT-BNNPTNT D...
9,203/2025/QH15,Resolution,AMENDMENTS TO CERTAIN ARTICLES OF THE CONSTITU...


Unresolved family candidates: 27


#### **4. Split complete families**

In [32]:
# Balance total size and document types without dividing families
def split_regulatory_families(families: dict, ratios: dict, seed: int) -> dict:
    rng = random.Random(seed)
    all_docs = [doc for family in families.values() for doc in family]
    all_types = Counter(doc.get("document_type", "Unknown") for doc in all_docs)

    target_sizes = {split: len(all_docs) * ratio for split, ratio in ratios.items()}
    target_types = {split: {doc_type: count * ratios[split] for doc_type, count in all_types.items()}
                    for split in ratios}

    assignments = {split: [] for split in ratios}
    current_sizes = Counter()
    current_types = {split: Counter() for split in ratios}

    family_items = list(families.items())
    rng.shuffle(family_items)
    family_items.sort(key=lambda item: len(item[1]), reverse=True)

    for family_id, family_docs in family_items:
        family_types = Counter(doc.get("document_type", "Unknown") for doc in family_docs)

        def score(candidate):
            total = 0

            for split in ratios:
                added_size = len(family_docs) if split == candidate else 0
                new_size = current_sizes[split] + added_size
                size_error = ((new_size - target_sizes[split]) / max(target_sizes[split], 1)) ** 2
                type_error = sum((current_types[split][doc_type]
                                  + (family_types[doc_type] if split == candidate else 0)
                                  - target_types[split][doc_type]) ** 2
                                 / max(target_types[split][doc_type], 1) for doc_type in all_types)
                overflow = max(0, new_size - target_sizes[split]) / max(target_sizes[split], 1)
                total += size_error + type_error + 4 * overflow

            return total

        selected_split = min(ratios, key=score)
        assignments[selected_split].append(family_id)
        current_sizes[selected_split] += len(family_docs)
        current_types[selected_split].update(family_types)

    return {split: [doc for family_id in family_ids for doc in families[family_id]]
            for split, family_ids in assignments.items()}


split_documents = split_regulatory_families(families, SPLIT_RATIOS, RANDOM_SEED)
dev_docs, eval_docs, test_docs = (split_documents["development"],
                                  split_documents["eval"], split_documents["test"])

print("Development:", len(dev_docs))
print("Evaluation:", len(eval_docs))
print("Test:", len(test_docs))
print("Total:", sum(len(items) for items in split_documents.values()))


Development: 283
Evaluation: 60
Test: 60
Total: 403


#### **5. Build and validate the split manifest**

In [33]:
# Extract a four-digit year from issue date or official number
def get_publication_year(doc: dict):
    match = re.search(r"\b(?:19|20)\d{2}\b",
                      f"{doc.get('issue_date', '')} {doc.get('official_number', '')}")
    return int(match.group()) if match else None


# Create one reproducible manifest record per document
manifest_records = []

for split_name, split_docs in split_documents.items():
    for doc in split_docs:
        manifest_records.append({
            "doc_id": doc["doc_id"],
            "official_number": doc.get("official_number", ""),
            "source_filename": doc["source_filename"],
            "cleaned_filename": Path(doc["cleaned_path"]).name,
            "split": split_name,
            "document_type": doc.get("document_type", "Unknown"),
            "hierarchy_level": doc.get("hierarchy_level", 99),
            "esg_domains": doc.get("esg_domains", []),
            "esg_categories": doc.get("esg_categories", []),
            "publication_year": get_publication_year(doc),
            "regulatory_family_id": doc["regulatory_family_id"],
            "random_seed": RANDOM_SEED
        })

manifest_df = pd.DataFrame(manifest_records)


In [34]:
# Confirm complete assignment and zero family leakage
assert len(manifest_df) == len(docs)
assert manifest_df["doc_id"].is_unique
assert manifest_df.groupby("regulatory_family_id")["split"].nunique().max() == 1

split_summary = manifest_df["split"].value_counts().rename("document_count").to_frame()
split_summary["percentage"] = (split_summary["document_count"] / len(manifest_df) * 100).round(2)

document_type_distribution = pd.crosstab(manifest_df["document_type"],
                                          manifest_df["split"], margins=True)
publication_year_distribution = pd.crosstab(manifest_df["publication_year"],
                                             manifest_df["split"], margins=True)

# Create one row per ESG domain and restore a unique index
esg_long = (manifest_df[["split", "esg_domains"]]
            .explode("esg_domains")
            .dropna(subset=["esg_domains"])
            .reset_index(drop=True))

esg_distribution = pd.crosstab(esg_long["esg_domains"],
                                esg_long["split"], margins=True)

print("Family leakage: 0")
print("Documents assigned:", len(manifest_df))
display(split_summary)
display(document_type_distribution)
display(esg_distribution)
display(publication_year_distribution)


Family leakage: 0
Documents assigned: 403


,document_count,percentage
split,,
development,283,70.22
eval,60,14.89
test,60,14.89


split,development,eval,test,All
document_type,,,,
Announcement,1,0,0,1
Circular,109,24,24,157
Decision,39,8,8,55
Decree,72,15,15,102
Directive,1,0,0,1
Integrated Document,4,1,1,6
Law,46,10,10,66
Notification,1,0,0,1
Order,1,0,0,1


split,development,eval,test,All
esg_domains,,,,
Environment,241,47,50,338
Governance,154,27,33,214
Social,97,19,27,143
All,492,93,110,695


split,development,eval,test,All
publication_year,,,,
1998.0,0,0,1,1
2001.0,1,0,0,1
2004.0,0,1,0,1
2006.0,0,1,0,1
2007.0,1,1,0,2
2008.0,2,0,0,2
2009.0,2,1,0,3
2010.0,1,1,1,3
2011.0,3,0,0,3


#### **6. Copy files and save outputs**

In [35]:
# Refresh each split folder so reruns do not leave stale files
def copy_split(documents: list, destination: Path):
    for old_file in destination.glob("*.txt"):
        old_file.unlink()

    for doc in documents:
        source = Path(doc["cleaned_path"])
        shutil.copy2(source, destination / source.name)


copy_split(dev_docs, SPLIT_DEV)
copy_split(eval_docs, SPLIT_EVAL)
copy_split(test_docs, SPLIT_TEST)

print("Split files copied:", len(dev_docs) + len(eval_docs) + len(test_docs))


Split files copied: 403


In [36]:
# Save the manifest and compact validation report
manifest = {
    "seed": RANDOM_SEED,
    "ratios": SPLIT_RATIOS,
    "counts": {split: len(items) for split, items in split_documents.items()},
    "family_counts": {"total": len(families), "singletons": singletons, "multi_document": multi},
    "documents": manifest_records
}

manifest_path = MANIFEST_FOLDER / "split_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

report_path = REPORT_FOLDER / "step1_3_split_report.txt"
report_text = "\n\n".join([
    "STEP 1.3 - DATASET SPLIT REPORT",
    split_summary.to_string(),
    "DOCUMENT TYPE DISTRIBUTION\n" + document_type_distribution.to_string(),
    "ESG DOMAIN DISTRIBUTION\n" + esg_distribution.to_string(),
    "PUBLICATION YEAR DISTRIBUTION\n" + publication_year_distribution.to_string(),
    "Family leakage: 0"
])
report_path.write_text(report_text, encoding="utf-8")

print("Manifest:", manifest_path)
print("Report:", report_path)


Manifest: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/manifests/split_manifest.json
Report: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/reports/step1_3_split_report.txt


In [3]:
# READ-ONLY CHECK:
# Confirm whether parser-inspection documents are in the development set

import json
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path(
    "/Users/tanggiee/Desktop/RAG_AI/esg_rag_project"
)

SPLIT_MANIFEST_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "manifests"
    / "split_manifest.json"
)

ORIGINAL_INSPECTION_MANIFEST = (
    PROJECT_ROOT
    / "outputs"
    / "article_inspection_samples"
    / "sample_manifest.csv"
)

EXPANDED_INSPECTION_MANIFEST = (
    PROJECT_ROOT
    / "outputs"
    / "article_inspection_expanded"
    / "expanded_sample_manifest.csv"
)


# Load the existing frozen split manifest
assert SPLIT_MANIFEST_PATH.exists(), (
    f"Split manifest not found: {SPLIT_MANIFEST_PATH}"
)

saved_manifest = json.loads(
    SPLIT_MANIFEST_PATH.read_text(encoding="utf-8")
)

manifest_records = (
    saved_manifest["documents"]
    if isinstance(saved_manifest, dict)
    else saved_manifest
)

saved_manifest_df = pd.DataFrame(manifest_records)


# Load filenames used during parser inspection
inspection_filenames = set()

for inspection_path in [
    ORIGINAL_INSPECTION_MANIFEST,
    EXPANDED_INSPECTION_MANIFEST,
]:
    if inspection_path.exists():
        inspection_df = pd.read_csv(inspection_path)

        assert "source_filename" in inspection_df.columns, (
            f"source_filename missing from {inspection_path}"
        )

        inspection_filenames.update(
            inspection_df["source_filename"]
            .dropna()
            .astype(str)
        )
    else:
        print(
            "Inspection manifest not found:",
            inspection_path
        )

assert inspection_filenames, (
    "No inspection filenames were loaded."
)


# Match inspection documents to the existing split
inspection_split_check = (
    saved_manifest_df[
        saved_manifest_df["source_filename"].isin(
            inspection_filenames
        )
    ][
        [
            "doc_id",
            "source_filename",
            "regulatory_family_id",
            "split",
            "document_type",
        ]
    ]
    .sort_values(
        [
            "split",
            "regulatory_family_id",
            "source_filename",
        ]
    )
    .reset_index(drop=True)
)


# Identify filenames not found in the split manifest
matched_filenames = set(
    inspection_split_check["source_filename"]
)

unmatched_filenames = sorted(
    inspection_filenames - matched_filenames
)

outside_development = inspection_split_check[
    ~inspection_split_check["split"].eq("development")
]


display(inspection_split_check)

print("\nInspection documents by split:")
print(
    inspection_split_check["split"]
    .value_counts(dropna=False)
)

print(
    "\nInspection documents outside development:",
    len(outside_development)
)

print(
    "Inspection filenames not matched:",
    len(unmatched_filenames)
)

display(outside_development)

if unmatched_filenames:
    display(
        pd.DataFrame(
            {"unmatched_source_filename": unmatched_filenames}
        )
    )

/Users/tanggiee/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/tanggiee/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,doc_id,source_filename,regulatory_family_id,split,document_type
0,01_2021_TT-BXD_m_488637,01_2021_TT-BXD_m_488637.docx,family_0002,development,Circular
1,07_2025_TT-BTNMT_m_647200,07_2025_TT-BTNMT_m_647200.docx,family_0005,development,Circular
2,08_2022_ND-CP_m_507203,08_2022_ND-CP_m_507203.docx,family_0005,development,Decree
3,08_2025_TT-BNNMT_m_666356,08_2025_TT-BNNMT_m_666356.docx,family_0005,development,Circular
4,110_2026_ND-CP_702140,110_2026_ND-CP_702140.docx,family_0005,development,Decree
5,11_VBHN-BTC_m_609280,11_VBHN-BTC_m_609280.docx,family_0005,development,Integrated Document
6,40_2026_ND-CP_m_695776,40_2026_ND-CP_m_695776.docx,family_0005,development,Decree
7,79_2023_ND-CP_m_590378,79_2023_ND-CP_m_590378.docx,family_0005,development,Decree
8,125_VBHN-VPQH_m_682303,125_VBHN-VPQH_m_682303.docx,family_0011,development,Integrated Document
9,146_2025_QH15_706223,146_2025_QH15_706223.docx,family_0011,development,Law



Inspection documents by split:
split
development    31
eval           13
test            8
Name: count, dtype: int64

Inspection documents outside development: 21
Inspection filenames not matched: 0


,doc_id,source_filename,regulatory_family_id,split,document_type
31,78_2025_QH15_683248,78_2025_QH15_683248.docx,family_0029,eval,Law
32,07_2017_QH14_m_355880,07_2017_QH14_m_355880.docx,family_0035,eval,Law
33,08_2020_QD-TTg_m_444837,08_2020_QD-TTg_m_444837.docx,family_0040,eval,Decision
34,24_2014_QD-TTg_m_226185,24_2014_QD-TTg_m_226185.docx,family_0040,eval,Decision
35,108_2025_ND-CP_m_658466,108_2025_ND-CP_m_658466.docx,family_0058,eval,Decree
36,144_2024_ND-CP_m_647300,144_2024_ND-CP_m_647300.docx,family_0058,eval,Decree
37,199_2025_ND-CP_m_664759,199_2025_ND-CP_m_664759.docx,family_0058,eval,Decree
38,138_NQ-CP_m_544532,138_NQ-CP_m_544532.docx,family_0081,eval,Resolution
39,356_2025_ND-CP_689146,356_2025_ND-CP_689146.docx,family_0101,eval,Decree
40,1690_QD-TTg_m_114738,1690_QD-TTg_m_114738.docx,family_0104,eval,Decision
